# Solutions 3 · Choose X or ln_X, then Split and Predict

Sample solutions for the quiz notebook `qz-03-machine-learning`. Your numbers may differ slightly if you choose other features.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/sol-03-machine-learning.ipynb)

- Lecture: [Chapter 3 · Machine Learning and Train–Test Split](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-03-machine-learning.md)
- Labs: [LM301](https://rathachai.github.io/DA-LAB/learnings/linear/lm301.html) | [LM302](https://rathachai.github.io/DA-LAB/learnings/linear/lm302.html)
- Demo notebook: [nb-03-machine-learning](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-03-machine-learning.ipynb)

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---

## How to use
1. Run the SETUP cell first. It loads the quiz dataset `df` from a CSV file.
2. For each exercise, read the blue box, then write your code in the empty cell(s) below it.
3. Use **pandas** for tables, **numpy** for calculations, **scikit-learn** for the model and metrics, and **matplotlib** for plots.

Sample solutions are given in the green boxes. Try the quiz notebook first, then compare.

1. รันเซลล์ SETUP ก่อน เซลล์นี้โหลดชุดข้อมูลควิซ `df` จากไฟล์ CSV
2. ในแต่ละแบบฝึกหัด ให้อ่านกล่องสีน้ำเงิน แล้วเขียนโค้ดในเซลล์ว่างด้านล่าง
3. ใช้ **pandas** สำหรับตาราง, **numpy** สำหรับการคำนวณ, **scikit-learn** สำหรับโมเดลและตัวชี้วัด และ **matplotlib** สำหรับกราฟ

เฉลยตัวอย่างอยู่ในกล่องสีเขียว ลองทำโน้ตบุ๊กควิซก่อน แล้วจึงเปรียบเทียบ

## SETUP
The quiz data are in `quiz03_data.csv`: **200 samples** with the columns `id`, five inputs `x1`–`x5` and the target `y`. Not every input is useful, and some inputs may work better after taking the natural log (`ln_x`).

ข้อมูลควิซอยู่ใน `quiz03_data.csv`: **200 ตัวอย่าง** มีคอลัมน์ `id`, อินพุตห้าตัว `x1`–`x5` และเป้าหมาย `y` อินพุตบางตัวไม่มีประโยชน์ และบางตัวอาจใช้ได้ดีขึ้นเมื่อใช้ลอการิทึมธรรมชาติ (`ln_x`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/quiz03_data.csv"
df = pd.read_csv(url)
print(df.shape)
df

### Exercise 1 · Look at the data

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 1.** Display `df.describe()`. Then compute the **correlation of every input `x1`–`x5` with `y`** and draw them as a bar chart. Also draw five small scatter plots (one for each `x` against `y`). Which inputs look useful? Which scatter plots look curved or squeezed to the left?<br>
*Hint: `df.describe()`, `df[features].corrwith(df["y"])`, `.plot.bar()`, `plt.subplots(1, 5)` and `ax.scatter`.*

**แบบฝึกหัด 1.** แสดงผล `df.describe()` จากนั้นคำนวณ**ค่าสหสัมพันธ์ (correlation) ของอินพุตทุกตัว `x1`–`x5` กับ `y`** และวาดเป็นแผนภูมิแท่ง วาดแผนภาพกระจาย (scatter plot) ขนาดเล็ก 5 รูป (แต่ละ `x` เทียบกับ `y`) อินพุตตัวใดดูมีประโยชน์ แผนภาพกระจายรูปใดดูโค้งหรืออัดไปทางซ้าย<br>
*คำใบ้: `df.describe()`, `df[features].corrwith(df["y"])`, `.plot.bar()`, `plt.subplots(1, 5)` และ `ax.scatter`*

</div>

In [ ]:
print(df.describe().round(2))

features = ["x1", "x2", "x3", "x4", "x5"]
corr = df[features].corrwith(df["y"])
print(corr.round(3))
corr.plot.bar(title="Correlation with y")
plt.axhline(0, color="k", lw=0.8); plt.show()

fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for ax, name in zip(axs, features):
    ax.scatter(df[name], df["y"], s=6)
    ax.set_xlabel(name)
axs[0].set_ylabel("y")
plt.tight_layout(); plt.show()

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** `x2` (r = 0.71) and `x1` (0.34) are useful, `x4` (-0.24) is weak, `x3` (0.08) and `x5` (-0.01) look useless. The scatter plots of `x2` and `x3` are squeezed to the left with a long tail to the right, and `y` against `x2` bends like a curve. That shape is a hint to try the natural log.

**เฉลย.** `x2` (r = 0.71) และ `x1` (0.34) มีประโยชน์, `x4` (-0.24) อ่อน, `x3` (0.08) และ `x5` (-0.01) ดูไม่มีประโยชน์ แผนภาพกระจายของ `x2` และ `x3` อัดไปทางซ้ายและมีหางยาวไปทางขวา และ `y` เทียบกับ `x2` โค้งเป็นเส้นโค้ง รูปร่างนี้เป็นสัญญาณให้ลองใช้ลอการิทึมธรรมชาติ

</div>

### Exercise 2 · Process the data and select X and y

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 2.** For the inputs that look curved, add a log column (for example `ln_x2 = np.log(df["x2"])`) and compare its correlation with `y` to the correlation of the raw column. Then **select your features** `X` (a list of columns, raw `x` or `ln_x`) and the target `y`. Explain in one line why you keep or drop each input.<br>
*Hint: `np.log`, `df[[...]]`. Remember that `id` is a label, not an input.*

**แบบฝึกหัด 2.** สำหรับอินพุตที่ดูโค้ง ให้เพิ่มคอลัมน์ลอการิทึม (เช่น `ln_x2 = np.log(df["x2"])`) แล้วเปรียบเทียบค่าสหสัมพันธ์กับ `y` ของคอลัมน์นี้กับคอลัมน์ดิบ จากนั้น**เลือกฟีเจอร์** `X` (รายการคอลัมน์ ใช้ `x` ดิบหรือ `ln_x`) และเป้าหมาย `y` อธิบายเหตุผลที่เก็บหรือตัดอินพุตแต่ละตัวอย่างละหนึ่งบรรทัด<br>
*คำใบ้: `np.log`, `df[[...]]` จำไว้ว่า `id` เป็นป้ายระบุแถว ไม่ใช่อินพุต*

</div>

In [ ]:
df["ln_x2"] = np.log(df["x2"])
df["ln_x3"] = np.log(df["x3"])

print("x2    :", round(df["x2"].corr(df["y"]), 3), "  ln_x2:", round(df["ln_x2"].corr(df["y"]), 3))
print("x3    :", round(df["x3"].corr(df["y"]), 3), "  ln_x3:", round(df["ln_x3"].corr(df["y"]), 3))

selected = ["x1", "ln_x2", "x4"]
X = df[selected]
y = df["y"]
print(X.shape, y.shape)

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** `ln_x2` has r = 0.876 against 0.707 for raw `x2`, so the log helps and `ln_x2` replaces `x2`. `ln_x3` stays near zero (0.07), so the log does not rescue `x3`. Selected: `x1`, `ln_x2`, `x4`. Dropped: `x3` and `x5` (no relation), raw `x2` (replaced by its log), `id` (a label).

**เฉลย.** `ln_x2` มี r = 0.876 เทียบกับ 0.707 ของ `x2` ดิบ ลอการิทึมจึงช่วยได้และ `ln_x2` มาแทน `x2` ส่วน `ln_x3` ยังใกล้ศูนย์ (0.07) ลอการิทึมจึงช่วย `x3` ไม่ได้ ที่เลือก: `x1`, `ln_x2`, `x4` ที่ตัด: `x3` และ `x5` (ไม่มีความสัมพันธ์), `x2` ดิบ (ถูกแทนด้วย log), `id` (เป็นป้ายระบุแถว)

</div>

### Exercise 3 · Split, train and predict

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 3.** Split the data **70 / 30** into train and test (`random_state=7`). Fit a `LinearRegression` on your selected features using the train part only, and predict `y_pred` for the test part. Print the equation (intercept and one coefficient per feature).<br>
*Hint: `train_test_split(X, y, test_size=0.3, random_state=7)`, `LinearRegression().fit(...)`, `model.predict(...)`.*

**แบบฝึกหัด 3.** แบ่งข้อมูลเป็นชุดฝึก (train) และชุดทดสอบ (test) แบบ **70 / 30** (`random_state=7`) ฝึก `LinearRegression` ด้วยฟีเจอร์ที่เลือกโดยใช้ชุดฝึกเท่านั้น แล้วทำนาย `y_pred` ของชุดทดสอบ พิมพ์สมการ (ค่าตัดแกน และสัมประสิทธิ์ของแต่ละฟีเจอร์)<br>
*คำใบ้: `train_test_split(X, y, test_size=0.3, random_state=7)`, `LinearRegression().fit(...)`, `model.predict(...)`*

</div>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("y_pred = %.2f" % model.intercept_ + "".join(f" {c:+.2f}*{n}" for c, n in zip(model.coef_, selected)))

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** 140 rows train the model and 60 rows test it. The equation is about y = 24.5 + 0.49 x1 + 13.6 ln(x2) - 0.86 x4, close to the rule used to create the data (25 + 0.5 x1 + 14 ln x2 - 0.9 x4).

**เฉลย.** ใช้ 140 แถวฝึกโมเดลและ 60 แถวทดสอบ สมการประมาณ y = 24.5 + 0.49 x1 + 13.6 ln(x2) - 0.86 x4 ใกล้เคียงกับกฎที่ใช้สร้างข้อมูล (25 + 0.5 x1 + 14 ln x2 - 0.9 x4)

</div>

### Exercise 4 · Evaluate and visualize

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 4.** Report **MAE, RMSE, MAPE and R²** for the train part and for the test part (print one value at a time). Then plot `y` (actual) against `y_pred` for the test part with the diagonal line `y = y_pred`. Is the test error much larger than the train error?<br>
*Hint: `mean_absolute_error`, `np.sqrt(mean_squared_error(...))`, `mean_absolute_percentage_error`, `r2_score`; `plt.plot(lim, lim)` for the diagonal.*

**แบบฝึกหัด 4.** รายงานค่า **MAE, RMSE, MAPE และ R²** ของชุดฝึกและชุดทดสอบ (พิมพ์ทีละค่า) จากนั้นพล็อต `y` (ค่าจริง) เทียบกับ `y_pred` ของชุดทดสอบ พร้อมเส้นทแยงมุม `y = y_pred` ความคลาดเคลื่อนของชุดทดสอบมากกว่าชุดฝึกมากหรือไม่<br>
*คำใบ้: `mean_absolute_error`, `np.sqrt(mean_squared_error(...))`, `mean_absolute_percentage_error`, `r2_score`; ใช้ `plt.plot(lim, lim)` วาดเส้นทแยงมุม*

</div>

In [ ]:
y_train_pred = model.predict(X_train)

mae = mean_absolute_error(y_train, y_train_pred)
print("train MAE  =", round(mae, 3))

rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
print("train RMSE =", round(rmse, 3))

mape = 100 * mean_absolute_percentage_error(y_train, y_train_pred)
print("train MAPE =", round(mape, 2), "%")

r2 = r2_score(y_train, y_train_pred)
print("train R2   =", round(r2, 3))

mae = mean_absolute_error(y_test, y_pred)
print("test  MAE  =", round(mae, 3))

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("test  RMSE =", round(rmse, 3))

mape = 100 * mean_absolute_percentage_error(y_test, y_pred)
print("test  MAPE =", round(mape, 2), "%")

r2 = r2_score(y_test, y_pred)
print("test  R2   =", round(r2, 3))

In [ ]:
plt.scatter(y_test, y_pred, s=10)
lim = [y.min(), y.max()]
plt.plot(lim, lim, "k--", label="y = y_pred")
plt.xlabel("actual y"); plt.ylabel("predicted y"); plt.legend(); plt.show()

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Train: MAE 2.28, R2 0.966. Test: MAE 2.43, RMSE 2.99, R2 0.954. The test error is only a little higher than the train error, so the model generalises. The points sit close to the diagonal.

**เฉลย.** Train: MAE 2.28, R2 0.966 Test: MAE 2.43, RMSE 2.99, R2 0.954 ความคลาดเคลื่อนของ test สูงกว่า train เพียงเล็กน้อย โมเดลจึงใช้งานได้ทั่วไป (generalise) จุดอยู่ใกล้เส้นทแยงมุม

</div>

### Exercise 5 · Does the choice of X matter?

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 5.** Repeat Exercises 3–4 for four feature sets: (a) all raw inputs `x1`–`x5`, (b) the raw columns you think are useful (without any log), (c) your selected set with the `ln_x` columns, (d) only the single best column. Use 20 different random splits (`random_state` = 0, 1, ..., 19) and report the **mean test MAE** of each set in a table.<br>
*Hint: loop over `seed in range(20)`; store the results in a list of dictionaries and build a `pd.DataFrame`; use `groupby("set")`.*

**แบบฝึกหัด 5.** ทำแบบฝึกหัด 3–4 ซ้ำกับฟีเจอร์ 4 ชุด ได้แก่ (a) อินพุตดิบทั้งหมด `x1`–`x5`, (b) คอลัมน์ดิบที่คิดว่ามีประโยชน์ (ไม่ใช้ log), (c) ชุดที่เลือกโดยใช้คอลัมน์ `ln_x`, (d) คอลัมน์เดียวที่ดีที่สุด ใช้การสุ่มแบ่งข้อมูล 20 ครั้ง (`random_state` = 0, 1, ..., 19) และรายงาน**ค่าเฉลี่ย MAE ของชุดทดสอบ**ของแต่ละชุดเป็นตาราง<br>
*คำใบ้: วนลูป `seed in range(20)`; เก็บผลเป็นลิสต์ของ dictionary แล้วสร้าง `pd.DataFrame`; ใช้ `groupby("set")`*

</div>

In [ ]:
sets = {"(a) all raw x1-x5": ["x1", "x2", "x3", "x4", "x5"],
        "(b) raw x1, x2, x4": ["x1", "x2", "x4"],
        "(c) selected x1, ln_x2, x4": ["x1", "ln_x2", "x4"],
        "(d) best single ln_x2": ["ln_x2"]}

rows = []
for seed in range(20):
    train_df, test_df = train_test_split(df, test_size=0.3, random_state=seed)
    for name, cols in sets.items():
        m = LinearRegression().fit(train_df[cols], train_df["y"])
        mae = mean_absolute_error(test_df["y"], m.predict(test_df[cols]))
        rows.append({"set": name, "test_MAE": mae})

pd.DataFrame(rows).groupby("set")["test_MAE"].agg(["mean", "std"]).round(2)

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Mean test MAE over 20 splits: (a) all raw 6.95, (b) raw x1, x2, x4 6.90, **(c) selected with ln_x2 2.36**, (d) best single column `ln_x2` 5.94. Adding useless inputs (a) hardly changes the error, but the **log is what makes the big difference**: the straight-line model cannot follow the curved `x2`.

**เฉลย.** ค่าเฉลี่ย MAE ของ test จาก 20 การแบ่ง: (a) ดิบทั้งหมด 6.95, (b) x1, x2, x4 ดิบ 6.90, **(c) ชุดที่เลือกพร้อม ln_x2 2.36**, (d) คอลัมน์เดียวที่ดีที่สุด `ln_x2` 5.94 การเพิ่มอินพุตที่ไม่มีประโยชน์ (a) แทบไม่เปลี่ยนความคลาดเคลื่อน แต่ **log คือสิ่งที่สร้างความต่างอย่างมาก**: โมเดลเส้นตรงตาม `x2` ที่โค้งไม่ได้

</div>

### Exercise 6 · Conclusion

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 6.** In three or four sentences: which inputs did you select and why? Which input needed a log, and what did the log change? How much better is your selected set than "all raw inputs"? Why do we measure the error on the test part?

**แบบฝึกหัด 6.** ตอบ 3–4 ประโยค: เลือกอินพุตตัวใดและเพราะเหตุใด อินพุตตัวใดต้องใช้ log และ log เปลี่ยนอะไร ชุดที่เลือกดีกว่า "อินพุตดิบทั้งหมด" มากเพียงใด เหตุใดจึงวัดความคลาดเคลื่อนจากชุดทดสอบ

</div>

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Your answer:** I selected `x1`, `ln_x2` and `x4` because they relate to `y`; `x3` and `x5` carry no information and were dropped. `x2` is curved against `y` (a long right tail), so its log (`ln_x2`) gave a higher correlation (0.88 against 0.71) and a straight line fits it. The selected set reaches a test MAE of about 2.4 against 6.9 for all raw inputs (about 66 % smaller). We measure the error on the test part because the train error only shows how well the model memorised the rows it has already seen; the test error shows how it will do on new data.

**คำตอบของคุณ:** เลือก `x1`, `ln_x2` และ `x4` เพราะสัมพันธ์กับ `y`; `x3` และ `x5` ไม่มีข้อมูลจึงตัดออก `x2` โค้งเมื่อเทียบกับ `y` (หางยาวทางขวา) ดังนั้น log ของมัน (`ln_x2`) ให้สหสัมพันธ์สูงกว่า (0.88 เทียบกับ 0.71) และเส้นตรงจึงเข้ากันได้ ชุดที่เลือกได้ MAE ของ test ประมาณ 2.4 เทียบกับ 6.9 ของอินพุตดิบทั้งหมด (ลดลงประมาณ 66 %) เราวัดความคลาดเคลื่อนจากชุด test เพราะความคลาดเคลื่อนของ train บอกเพียงว่าโมเดลท่องจำแถวที่เคยเห็นได้ดีเพียงใด ส่วนความคลาดเคลื่อนของ test บอกว่าโมเดลจะทำงานกับข้อมูลใหม่ได้ดีเพียงใด

</div>

## Done
Compare your answers with the demo notebook and discuss them in class.

เปรียบเทียบคำตอบกับโน้ตบุ๊กตัวอย่าง (demo) และร่วมอภิปรายในชั้นเรียน

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---